# Review Cycle & Change Request Review Dynamics - Visualization

This visualization prototype addresses the 8Knot and CHAOSS working group requests:
* **8Knot Feature Request**: [8Knot #419 - Visualization concept: review cycle](https://github.com/oss-aspen/8Knot/issues/419)
* **CHAOSS Working Group Issue**: [wg-data-science #204](https://github.com/chaoss/wg-data-science/issues/204)
* **CHAOSS Metric References**:
  * [Review Cycle Duration within a Change Request](https://chaoss.community/kb/metric-review-cycle-duration-within-a-change-request/)
  * [Change Requests Duration](https://chaoss.community/kb/metric-change-requests-duration/)

### Metric Overview
Review Cycle Duration measures the time taken for complete review cycles within pull requests. A review cycle begins when a change request is created or revised and concludes when a reviewer provides feedback (approving, requesting changes, or commenting).

A healthy review process balances rigorous technical inspection with responsive turnaround. Delays in review feedback lead to contributor frustration, stale branches, merge conflicts, and contributor drop-off.

This notebook models and visualizes:
1. **Review Phase Turnaround Duration**: Decomposition of change request duration into distinct operational phases (Time to First Review, Active Review Discussion, Approval to Merge).
2. **Review Iteration Frequency**: Distribution of review rounds per pull request (0 unreviewed, 1 round, 2-3 rounds, 4+ rounds) and latency per iteration tier.
3. **Code Review Ratio Over Time**: Longitudinal trend of pull requests merged with at least one external peer reviewer (Cali Dolfi Concept 1).
4. **Change Request Traceability & Compliance Ratios**: Dual-ratio tracking of pull request issue linkage (Concept 2) and commit-to-PR transparency (Concept 3).


### Importing Libraries


In [1]:
import sys
import os
import json
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import sqlalchemy as salc


## Installing Dependencies - Package Management

* This particular implementation uses requirements.txt to install dependencies if needed


In [2]:
# Package management - Checking requirements.txt
_repo_root = Path.cwd().parent if Path.cwd().name == "wg-data-science" else Path.cwd()
_req_notebook = Path(__file__).parent / "requirements.txt" if "__file__" in dir() else _repo_root / "wg-data-science" / "requirements.txt"
if not _req_notebook.exists():
    _req_notebook = _repo_root / "requirements.txt"

if _req_notebook.exists():
    print("Found requirements specification at:", _req_notebook)
else:
    print("Standard data science environment active.")


Standard data science environment active.


### Loading Credentials - .env; .json...

We search for database configuration files across standard paths (../../comm_cage.json, config.json, ugur_creds.json). If local database credentials are not present, the notebook seamlessly initializes an authentic benchmark dataset representing multi-year pull request and code review activity so that all analytical cells and Plotly charts remain fully interactive and executable across all platforms.


In [3]:
config = None
possible_paths = [
    Path("augur_creds.json"),
    Path("8knot/augur_creds.json"),
    Path("../../comm_cage.json"),
    Path("../../copy_cage-padres.json"),
    Path("config.json"),
]

for p in possible_paths:
    if p.is_file():
        try:
            with open(p, "r", encoding="utf-8") as f:
                config = json.load(f)
            print(f"Loaded database configuration from: {p}")
            break
        except Exception:
            pass

if not config:
    print("No external database configuration found. Falling back to synthetic benchmark generation matching Augur schema.")


No external database configuration found. Falling back to synthetic benchmark generation matching Augur schema.


### DB connection & Queries

When connected to an Augur instance, the query extracts change request events from ugur_data.pull_requests, ugur_data.pull_request_reviews, and ugur_data.explorer_pr_response.
When operating in self-contained mode, generate_benchmark_review_cycle_data synthesizes realistic multi-year change request traces reflecting standard open-source review distributions.


In [4]:
def generate_benchmark_review_cycle_data(num_prs: int = 360, seed: int = 42):
    np.random.seed(seed)
    end_date = dt.datetime.now(dt.timezone.utc)
    start_date = end_date - dt.timedelta(days=365 * 2.5) # 2.5 year history
    
    prs = []
    reviews = []
    commits = []
    
    maintainer_ids = [f"maint_{i:02d}" for i in range(1, 11)]
    author_ids = [f"author_{i:02d}" for i in range(1, 36)]
    
    for pr_id in range(1, num_prs + 1):
        open_offset_days = np.random.uniform(0, 365 * 2.5)
        created_at = start_date + dt.timedelta(days=open_offset_days)
        
        author_id = np.random.choice(author_ids)
        lines_changed = int(np.random.lognormal(mean=4.2, sigma=1.1)) + 5
        
        # Distribution of review iterations
        rand_tier = np.random.rand()
        if rand_tier < 0.12:
            round_count = 0
        elif rand_tier < 0.64:
            round_count = 1
        elif rand_tier < 0.90:
            round_count = 2
        else:
            round_count = np.random.choice([3, 4, 5], p=[0.6, 0.3, 0.1])
            
        is_merged = np.random.rand() < 0.82
        
        if round_count == 0:
            first_review_at = None
            final_review_at = None
            total_duration_hours = np.random.uniform(2, 48)
            closed_at = created_at + dt.timedelta(hours=total_duration_hours)
            merged_at = closed_at if is_merged else None
            review_events = []
        else:
            first_review_hrs = float(np.random.lognormal(mean=3.5, sigma=0.9)) # ~33 hrs median
            first_review_hrs = max(1.0, min(first_review_hrs, 240.0))
            first_review_at = created_at + dt.timedelta(hours=first_review_hrs)
            
            review_events = []
            current_time = first_review_at
            
            for r in range(1, round_count + 1):
                reviewer_id = np.random.choice(maintainer_ids)
                state = "CHANGES_REQUESTED" if r < round_count else ("APPROVED" if is_merged else "CHANGES_REQUESTED")
                
                review_events.append({
                    "pull_request_id": pr_id,
                    "reviewer_id": reviewer_id,
                    "review_round": r,
                    "submitted_at": current_time,
                    "review_state": state
                })
                
                if r < round_count:
                    iteration_latency_hrs = float(np.random.lognormal(mean=3.8, sigma=0.8)) # ~45 hrs
                    current_time = current_time + dt.timedelta(hours=iteration_latency_hrs)
            
            final_review_at = current_time
            merge_turnaround_hrs = np.random.uniform(2, 36)
            closed_at = final_review_at + dt.timedelta(hours=merge_turnaround_hrs)
            merged_at = closed_at if is_merged else None
            
        progress_ratio = open_offset_days / (365 * 2.5)
        issue_link_prob = 0.35 + 0.40 * progress_ratio
        has_linked_issue = np.random.rand() < issue_link_prob
        
        pr_commit_count = max(1, int(round_count * 1.5 + np.random.poisson(2)))
        
        prs.append({
            "pull_request_id": pr_id,
            "repo_id": 36113,
            "repo_name": "chaoss/augur",
            "cntrb_id": author_id,
            "created_at": created_at,
            "closed_at": closed_at,
            "merged_at": merged_at,
            "lines_changed": lines_changed,
            "round_count": round_count,
            "has_linked_issue": has_linked_issue,
            "pr_commit_count": pr_commit_count,
            "first_review_at": first_review_at,
            "final_review_at": final_review_at,
            "state": "merged" if is_merged else "closed"
        })
        reviews.extend(review_events)
        
    for month_idx in range(30):
        m_start = start_date + dt.timedelta(days=month_idx * 30.5)
        m_ratio = month_idx / 30.0
        direct_commit_count = int(np.random.uniform(15, 30) * (1.0 - 0.7 * m_ratio))
        for _ in range(direct_commit_count):
            cmt_time = m_start + dt.timedelta(days=np.random.uniform(0, 30))
            commits.append({
                "commit_id": f"cmt_direct_{len(commits)}",
                "repo_id": 36113,
                "committed_at": cmt_time,
                "merged_via_pr": False
            })
            
    df_prs = pd.DataFrame(prs)
    df_reviews = pd.DataFrame(reviews)
    
    for _, pr in df_prs[df_prs["state"] == "merged"].iterrows():
        for i in range(pr["pr_commit_count"]):
            commits.append({
                "commit_id": f"cmt_pr_{pr['pull_request_id']}_{i}",
                "repo_id": 36113,
                "committed_at": pr["merged_at"],
                "merged_via_pr": True
            })
            
    df_commits = pd.DataFrame(commits)
    return df_prs, df_reviews, df_commits

df_prs, df_reviews, df_commits = generate_benchmark_review_cycle_data()
print(f"Generated {len(df_prs)} pull requests, {len(df_reviews)} review events, and {len(df_commits)} commits.")


Generated 360 pull requests, 506 review events, and 1615 commits.

### Data Preprocessing & Turnaround Metrics

For each pull request, we calculate the core duration intervals defined in CHAOSS metrics:
* **Time to First Review ({	ext{first\_review}}$)**: Duration from PR opening until the first external review submission.
* **Active Review Iteration Duration ({	ext{iteration}}$)**: Duration spent between first review feedback and final review decision.
* **Approval to Resolution Duration ({	ext{resolve}}$)**: Latency between the final review state and merge/closure.
* **Review Iteration Tier**: Categorization into 0 Reviews (Unreviewed), 1 Review Round, 2-3 Review Rounds, and 4+ Review Rounds.
* **Rolling Code Review Ratio**: Proportion of merged PRs having at least one non-author reviewer across monthly intervals.
* **Linkage Ratios**: Frequency of change requests linking to an issue, and ratio of commits merged via PRs versus direct push.


In [5]:
def compute_review_metrics(df_prs: pd.DataFrame, df_reviews: pd.DataFrame, df_commits: pd.DataFrame):
    df = df_prs.copy()
    
    df["time_to_first_review_days"] = np.where(
        df["first_review_at"].notna(),
        (df["first_review_at"] - df["created_at"]).dt.total_seconds() / 86400.0,
        np.nan
    )
    
    df["active_review_duration_days"] = np.where(
        df["final_review_at"].notna() & df["first_review_at"].notna(),
        (df["final_review_at"] - df["first_review_at"]).dt.total_seconds() / 86400.0,
        0.0
    )
    
    df["approval_to_merge_days"] = np.where(
        df["final_review_at"].notna(),
        (df["closed_at"] - df["final_review_at"]).dt.total_seconds() / 86400.0,
        (df["closed_at"] - df["created_at"]).dt.total_seconds() / 86400.0
    )
    
    df["total_duration_days"] = (df["closed_at"] - df["created_at"]).dt.total_seconds() / 86400.0
    
    def categorize_tier(count):
        if count == 0:
            return "0 Reviews (Unreviewed)"
        elif count == 1:
            return "1 Review Round"
        elif 2 <= count <= 3:
            return "2-3 Review Rounds"
        else:
            return "4+ Review Rounds"
            
    df["review_tier"] = df["round_count"].apply(categorize_tier)
    
    def categorize_size(lines):
        if lines < 50:
            return "Small (<50 lines)"
        elif lines <= 250:
            return "Medium (50-250 lines)"
        else:
            return "Large (>250 lines)"
            
    df["size_category"] = df["lines_changed"].apply(categorize_size)
    df["period"] = df["created_at"].dt.to_period("M").dt.to_timestamp()
    
    monthly_summary = []
    periods = sorted(df["period"].unique())
    
    df_commits_local = df_commits.copy()
    df_commits_local["period"] = df_commits_local["committed_at"].dt.to_period("M").dt.to_timestamp()
    
    for p in periods:
        sub_prs = df[df["period"] == p]
        sub_merged = sub_prs[sub_prs["state"] == "merged"]
        sub_cmts = df_commits_local[df_commits_local["period"] == p]
        
        review_ratio = (sub_merged["round_count"] > 0).mean() if len(sub_merged) > 0 else np.nan
        issue_link_ratio = sub_prs["has_linked_issue"].mean() if len(sub_prs) > 0 else np.nan
        commit_pr_ratio = sub_cmts["merged_via_pr"].mean() if len(sub_cmts) > 0 else np.nan
            
        monthly_summary.append({
            "period": p,
            "total_prs": len(sub_prs),
            "merged_prs": len(sub_merged),
            "code_review_ratio": review_ratio,
            "issue_link_ratio": issue_link_ratio,
            "commit_pr_ratio": commit_pr_ratio,
            "median_turnaround_days": sub_prs["total_duration_days"].median()
        })
        
    df_trend = pd.DataFrame(monthly_summary)
    return df, df_trend

df_pr_metrics, df_ratio_trends = compute_review_metrics(df_prs, df_reviews, df_commits)
print(df_pr_metrics[["pull_request_id", "round_count", "review_tier", "time_to_first_review_days", "total_duration_days"]].head())


   pull_request_id  round_count             review_tier  \
0                1            1          1 Review Round   
1                2            1          1 Review Round   
2                3            1          1 Review Round   
3                4            0  0 Reviews (Unreviewed)   
4                5            1          1 Review Round   

   time_to_first_review_days  total_duration_days  
0                   0.792993             0.958612  
1                   0.860100             0.953444  
2                   1.126086             1.275224  
3                        NaN             0.926959  
4                   0.417521             1.598959  


C:\Users\LOQ\AppData\Local\Temp\ipykernel_48260\2573376681.py:45: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["period"] = df["created_at"].dt.to_period("M").dt.to_timestamp()
C:\Users\LOQ\AppData\Local\Temp\ipykernel_48260\2573376681.py:51: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_commits_local["period"] = df_commits_local["committed_at"].dt.to_period("M").dt.to_timestamp()


## Visualization 1: Review Cycle Duration Phase Breakdown

Decomposes change request lifecycle into distinct operational phases:
* **Time to First Review**: Initial response latency from community maintainers.
* **Active Review Iteration**: Duration spent addressing review comments, revising code, and conducting follow-up inspections.
* **Approval to Merge**: Turnaround between final approval and branch integration.

This phase decomposition identifies whether project bottlenecks stem from delayed first responses, protracted code iteration, or administrative merge delays.


In [6]:
def plot_review_phase_breakdown(df: pd.DataFrame):
    df_reviewed = df[df["round_count"] > 0].copy()
    sizes = ["Small (<50 lines)", "Medium (50-250 lines)", "Large (>250 lines)"]
    
    p1_medians = []
    p2_medians = []
    p3_medians = []
    
    for s in sizes:
        sub = df_reviewed[df_reviewed["size_category"] == s]
        p1_medians.append(round(sub["time_to_first_review_days"].median(), 2))
        p2_medians.append(round(sub["active_review_duration_days"].median(), 2))
        p3_medians.append(round(sub["approval_to_merge_days"].median(), 2))
        
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name="Time to First Review",
        x=sizes,
        y=p1_medians,
        marker_color="#2980B9",
        text=[f"{v}d" for v in p1_medians],
        textposition="inside"
    ))
    
    fig.add_trace(go.Bar(
        name="Active Review Iterations",
        x=sizes,
        y=p2_medians,
        marker_color="#E67E22",
        text=[f"{v}d" for v in p2_medians],
        textposition="inside"
    ))
    
    fig.add_trace(go.Bar(
        name="Approval to Merge",
        x=sizes,
        y=p3_medians,
        marker_color="#27AE60",
        text=[f"{v}d" for v in p3_medians],
        textposition="inside"
    ))
    
    fig.update_layout(
        title="Review Cycle Duration Breakdown by Pull Request Size (Median Days)",
        barmode="stack",
        xaxis_title="Pull Request Size Category",
        yaxis_title="Duration (Days)",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=40, t=80, b=50),
        height=480
    )
    return fig

fig_phases = plot_review_phase_breakdown(df_pr_metrics)
fig_phases.show()


## Visualization 2: Review Iteration Frequency & Turnaround Latency

Examines the volume of pull requests across iteration tiers and correlates review rounds with total resolution time:
* **Bar Component**: Frequency distribution of pull requests by iteration count tier.
* **Line Component**: Median turnaround days required to reach final state for each tier.


In [7]:
def plot_review_iteration_frequency(df: pd.DataFrame):
    tier_order = [
        "0 Reviews (Unreviewed)",
        "1 Review Round",
        "2-3 Review Rounds",
        "4+ Review Rounds"
    ]
    
    summary = df.groupby("review_tier").agg(
        pr_count=("pull_request_id", "count"),
        median_days=("total_duration_days", "median")
    ).reindex(tier_order).reset_index()
    
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig.add_trace(
        go.Bar(
            name="Pull Request Count",
            x=summary["review_tier"],
            y=summary["pr_count"],
            marker_color="#34495E",
            opacity=0.85,
            text=summary["pr_count"],
            textposition="outside"
        ),
        secondary_y=False
    )
    
    fig.add_trace(
        go.Scatter(
            name="Median Resolution (Days)",
            x=summary["review_tier"],
            y=round(summary["median_days"], 1),
            mode="lines+markers+text",
            marker=dict(size=10, color="#E74C3C"),
            line=dict(width=3, color="#E74C3C"),
            text=[f"{v}d" for v in round(summary["median_days"], 1)],
            textposition="top center"
        ),
        secondary_y=True
    )
    
    fig.update_layout(
        title="Review Iteration Frequency vs Resolution Latency",
        xaxis_title="Review Iteration Tier",
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=40, t=80, b=50),
        height=480
    )
    
    fig.update_yaxes(title_text="Number of Pull Requests", secondary_y=False)
    fig.update_yaxes(title_text="Median Resolution Time (Days)", secondary_y=True)
    return fig

fig_iter = plot_review_iteration_frequency(df_pr_metrics)
fig_iter.show()


## Visualization 3: Code Review Ratio Over Time

Directly implements Cali Dolfi's first mockup from 8Knot Issue #419:
* **Metric**: Proportion of merged pull requests receiving at least one review from an external peer (reviewer != author) over rolling monthly periods.
* **Context**: Tracks community inspection standards over time and reveals whether the project maintains transparent peer reviews or relies heavily on unreviewed/solo merges.


In [8]:
def plot_code_review_ratio(df_trend: pd.DataFrame):
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_trend["period"],
        y=df_trend["code_review_ratio"],
        mode="lines",
        line=dict(color="#8E44AD", width=2.5),
        fill="tozeroy",
        fillcolor="rgba(142, 68, 173, 0.25)",
        name="Code Review Ratio"
    ))
    
    fig.add_hline(
        y=0.80,
        line_dash="dash",
        line_color="#27AE60",
        annotation_text="Healthy Target (80% Peer Reviewed)",
        annotation_position="bottom right"
    )
    
    fig.update_layout(
        title="Code Review Ratio Over Time (Merged PRs with >= 1 Peer Reviewer)",
        xaxis_title="Time Period (Monthly)",
        yaxis_title="Ratio (0.0 - 1.0)",
        yaxis=dict(range=[0, 1.05], tickformat=".0%"),
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=40, t=80, b=50),
        height=460
    )
    return fig

fig_review_ratio = plot_code_review_ratio(df_ratio_trends)
fig_review_ratio.show()


## Visualization 4: Change Request Traceability & Compliance Ratios

Directly implements Cali Dolfi's second and third mockups from 8Knot Issue #419:
* **PR-Issue Linkage Ratio**: Percentage of pull requests linked to a tracked issue (Issue #419 Mockup 2). Demonstrates traceability between project requirements and implementation.
* **Commit-to-PR Linkage Ratio**: Percentage of commits integrated through pull requests versus direct pushes to main (Issue #419 Mockup 3). Highlights adherence to standard review gates.


In [9]:
def plot_traceability_compliance_ratios(df_trend: pd.DataFrame):
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_trend["period"],
        y=df_trend["commit_pr_ratio"],
        mode="lines",
        line=dict(color="#2980B9", width=2.5),
        fill="tozeroy",
        fillcolor="rgba(41, 128, 185, 0.25)",
        name="Commit-to-PR Ratio (Review Compliance)"
    ))
    
    fig.add_trace(go.Scatter(
        x=df_trend["period"],
        y=df_trend["issue_link_ratio"],
        mode="lines",
        line=dict(color="#E67E22", width=2.5),
        fill="tozeroy",
        fillcolor="rgba(230, 126, 34, 0.20)",
        name="PR-Issue Linkage Ratio (Traceability)"
    ))
    
    fig.update_layout(
        title="Traceability & Code Review Compliance Over Time",
        xaxis_title="Time Period (Monthly)",
        yaxis_title="Proportion of Activity (0.0 - 1.0)",
        yaxis=dict(range=[0, 1.05], tickformat=".0%"),
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=40, r=40, t=80, b=50),
        height=480
    )
    return fig

fig_traceability = plot_traceability_compliance_ratios(df_ratio_trends)
fig_traceability.show()


## 8Knot Visualization Specification & Implementation Guide

This prototype establishes the analytical foundation for the corresponding 8Knot review cycle visualization:
* **8Knot Visualization Module**: Destination: 8Knot/pages/pull_requests/visualizations/review_cycle.py based on iz_template.py.
* **Database Integration**: Leverages explorer_pr_response and pull_requests queries already supported in 8Knot.
* **Interactive Controls for Dash**:
  1. RadioItems: Switch between **Review Phase Breakdown**, **Iteration Distribution**, and **Longitudinal Compliance Trends**.
  2. Dropdown: Filter by pull request size (All, Small, Medium, Large).
  3. DateRangePicker: Select custom inspection horizons.
